### Name : Hamza Nisar
### ID: FA23-BSCS-0163
### Course : AI-LAB 

# AI Interview Preparation Agent
### Built with CrewAI + Ollama (Llama 3) + SerperDevTool

This project is a **multi-agent AI system** that researches any job role and creates a complete, personalized interview preparation plan for it.

Two agents work in sequence: an **Interview Researcher** that finds required skills, common questions, and key topics for the given job role, and an **Interview Coach** that turns those findings into a structured 7-day preparation plan with practice questions and final tips.

**Test Input 1:** Software Engineer
**Test Input 2:** Data Analyst

## ⚙️ Part 1: Environment Setup

A **virtual environment** isolates this project's packages from your global Python installation, preventing version conflicts with other projects on your machine.

Upgrading `pip` first ensures the latest version of Python's package manager is used, which avoids random installation errors that older versions sometimes cause.

Installs the three core libraries: **crewai** (the multi-agent framework), **crewai-tools** (built-in tools like web search), and **python-dotenv** (to load API keys safely from a `.env` file).

Installs **langchain-ollama**, the bridge library that lets Python communicate with locally running Ollama models (like Llama 3) using a simple and familiar interface.

Installs **LiteLLM**, a unified interface that allows CrewAI to communicate with many different LLM providers (OpenAI, Ollama, Gemini, etc.) using the same syntax.

Installs **ipykernel**, which adds Jupyter notebook support to this virtual environment so it can be selected as a kernel inside VS Code or JupyterLab.

Registers this virtual environment as a named, selectable Jupyter kernel so the notebook runs using exactly these installed packages instead of the system default Python.

✅ **Done!** Select the **"Python (CrewAI Lab)"** kernel from the top-right of this notebook before running any code cells below.

## 🦙 Part 2: Ollama Setup

**Ollama** runs large language models like Llama 3 **fully locally on your own machine** — no internet, no cloud cost, and no API key needed for the model itself.

Download and install Ollama for your operating system from https://ollama.com/download. After installing, the next terminal command verifies the installation worked correctly.

Downloads the **Llama 3** model (a few GB) into your local Ollama environment. `ollama list` then confirms the model is available and ready to use offline.

Starts the Ollama server in the background — CrewAI and LangChain connect to it over `localhost:11434` to send prompts and receive responses from the local model.

> ⚠️ **Note:** If you see `bind: Only one usage of each socket address`, Ollama is **already running** in the background — skip `ollama serve` and continue to the next cell.

##  Part 3: CrewAI Core Concepts & Connecting to Llama 3

This cell simply confirms that `crewai` and `crewai_tools` are installed correctly and importable before we build anything on top of them.

In [3]:
import crewai
import crewai_tools

print("CrewAI setup working ✅")

CrewAI setup working ✅


Imports `os` for environment variable access, and brings in the five core CrewAI components needed to build and run a complete multi-agent pipeline.

In [4]:
import os
from crewai import Agent, Task, Crew, Process, LLM

###  CrewAI Core Components

- **Agent** — an AI role with its own `role`, `goal`, and `backstory` that guides how the LLM behaves. Example: an "Interview Researcher" agent searches the web; a "Coach" agent creates a preparation plan.
- **Task** — a specific job assigned to one agent, with a clear `description` and `expected_output` format.
- **Crew** — packages agents + tasks together and runs them in a defined order (`Process.sequential` means one after another).

Imports `OllamaLLM` from LangChain and sends one test prompt to confirm Llama 3 is running locally and responding correctly before we build agents on top of it.

In [5]:
from langchain_ollama import OllamaLLM

ollama_llm_test = OllamaLLM(model="llama3", temperature=0.2)

response = ollama_llm_test.invoke("What are the top 3 skills every Software Engineer must have?")

print(response)

While the specific skills required of a software engineer can vary depending on the company, role, and technology stack, here are three essential skills that are widely considered crucial for any software engineer:

1. **Programming Fundamentals**:
	* Proficiency in one or more programming languages (e.g., Java, Python, C++, JavaScript).
	* Understanding of data structures, algorithms, and software design patterns.
	* Ability to write clean, readable, and maintainable code.
2. **Problem-Solving and Debugging**:
	* Strong analytical and problem-solving skills to identify and troubleshoot issues in code.
	* Experience with debugging tools and techniques (e.g., print statements, debuggers, logging).
	* Ability to break down complex problems into smaller, manageable parts and develop effective solutions.
3. **Communication and Collaboration**:
	* Effective communication skills to explain technical concepts to both technical and non-technical stakeholders.
	* Ability to work collaboratively

###  Temperature Effect Demo
Runs the same interview-themed prompt at **low temperature** (0.2 → focused, consistent) and **high temperature** (1.0 → creative, varied) to show how this setting affects the model's output style.

In [6]:
llm_low = OllamaLLM(model="llama3", temperature=0.2)
llm_high = OllamaLLM(model="llama3", temperature=1.0)

prompt = "Give one unique tip to crack a technical interview, in one sentence."

low_response = llm_low.invoke(prompt)
high_response = llm_high.invoke(prompt)

print("Low Temperature (0.2):", low_response)
print("High Temperature (1.0):", high_response)

Low Temperature (0.2): To crack a technical interview, focus on asking thoughtful and insightful questions that demonstrate your understanding of the company's technology stack and challenges, rather than just trying to answer every question perfectly.
High Temperature (1.0): To crack a technical interview, make sure you can explain the thought process behind your coding solution, rather than just spewing out code, by using the "I" statement format, for example, "I first considered the problem as a [insert clever analogy here], then I implemented [specific algorithm/data structure] to achieve [desired outcome]."


Creates the `LLM` object that CrewAI agents will actually use — this is CrewAI's own wrapper routed through LiteLLM to the local Ollama server running Llama 3.

In [7]:
os.environ["CREWAI_OPT_OUT_TELEMETRY"] = "1"
os.environ["OPENAI_API_KEY"] = "dummy"   # CrewAI requires this variable to exist even though we use Ollama

ollama_llm = LLM(
    model="ollama/llama3",
    base_url="http://localhost:11434",
    temperature=0.3,
    is_litellm=True
)

print("✅ CrewAI LLM ready")

✅ CrewAI LLM ready


##  Part 4: Demo A — Interview Tips Generator (no web search)

A simple warm-up demo using **only the model's own trained knowledge** (no live search). Two agents work together to generate general interview tips and a quick checklist for any given job role.

Defines two warm-up agents: a `tips_researcher` that lists general interview tips for a job role, and a `tips_writer` that formats those tips into a clean, readable checklist.

In [8]:
tips_researcher = Agent(
    role="Interview Tips Researcher",
    goal=(
        "For the given {job_role}, list 6-8 important general interview tips "
        "covering communication, body language, and common mistakes to avoid."
    ),
    backstory=(
        "You are an experienced HR professional who has conducted hundreds of "
        "interviews and knows exactly what interviewers look for in candidates."
    ),
    tools=[],
    llm=ollama_llm,
    verbose=True,
    allow_delegation=False,
)

tips_writer = Agent(
    role="Interview Checklist Writer",
    goal=(
        "Take the researcher's tips and format them into a clean, "
        "actionable pre-interview checklist for {job_role}."
    ),
    backstory=(
        "You are a career coach who specializes in helping candidates prepare "
        "confidently and present themselves professionally in interviews."
    ),
    tools=[],
    llm=ollama_llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Demo A agents ready")

✅ Demo A agents ready


Defines what each Demo A agent must produce: the researcher lists tips, and the writer formats them into a structured pre-interview checklist with clear action items.

In [9]:
tips_research_task = Task(
    description=(
        "For the job role {job_role}, list 6-8 important interview tips covering "
        "communication skills, body language, common mistakes, and mindset."
    ),
    expected_output="A bullet list of 6-8 interview tips with a 1-line explanation each.",
    agent=tips_researcher
)

tips_write_task = Task(
    description=(
        "Using the researcher's tips, create a structured pre-interview checklist "
        "for {job_role} with sections: Before the Interview, During the Interview, After the Interview."
    ),
    expected_output="A structured markdown pre-interview checklist.",
    agent=tips_writer
)

print("✅ Demo A tasks ready")

✅ Demo A tasks ready


Packages both agents and tasks into a `Crew` with `Process.sequential` so the researcher always finishes first before the writer begins formatting the checklist.

In [10]:
def build_tips_crew(job_role):
    return Crew(
        agents=[tips_researcher, tips_writer],
        tasks=[tips_research_task, tips_write_task],
        process=Process.sequential
    )

Runs Demo A on a sample job role to verify the no-search pipeline works end-to-end and both agents produce clean, readable output before moving to the main agent.

In [11]:
demo_role = input("Enter a job role (or press Enter for 'Python Developer'): ").strip() or "Python Developer"

tips_crew = build_tips_crew(demo_role)
tips_result = await tips_crew.kickoff_async(inputs={"job_role": demo_role})

print("\n===== PRE-INTERVIEW CHECKLIST =====\n")
print(getattr(tips_result, "final_output", tips_result))

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Tips Researcher                                                                               │
│                                                                                                                 │
│  Task: For the job role Python Developer, list 6-8 important interview tips covering communication skills,      │
│  body language, common mistakes, and mindset.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Tips Researcher                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are 6-8 important general interview tips covering communication skills, body language, common mistakes    │
│  to avoid, and mindset that can help Python Developers stand out during an interview:                           │
│                                                                                                                 │
│  • **Be prepared to explain complex concepts simply**: As a Python Developer, you'll be expected to             │
│  communicate technical ideas to non-technical stakeholders. Practice explaining complex concepts in simple      │
│  terms to demonstrate your ability to articulate technical information.                                         │
│                                                                                                                 │
│  • **Maintain eye contact and engage with the interviewer**: Eye contact is crucial for building trust and      │
│  showing interest. Make sure to maintain eye contact while answering questions and show enthusiasm for the      │
│  role and company.                                                                                              │
│                                                                                                                 │
│  • **Use positive body language**: Sit up straight, uncross your arms, and avoid fidgeting. These nonverbal     │
│  cues can convey confidence and engagement, which are essential qualities for a Python Developer.               │
│                                                                                                                 │
│  • **Avoid jargon and technical buzzwords**: While it's natural to use technical terms in your field, using     │
│  too much jargon or buzzwords can make you seem arrogant or out of touch with the interviewer's level of        │
│  understanding. Strive for clarity and simplicity in your communication.                                        │
│                                                                                                                 │
│  • **Show enthusiasm and interest in the company and role**: Demonstrate your passion for Python development    │
│  and the company's mission. Ask thoughtful questions about the role, team, and company culture to show your     │
│  engagement and motivation.                                                                                     │
│                                                                                                                 │
│  • **Don't apologize or downplay your skills**: As a Python Developer, you have valuable skills that can        │
│  benefit the organization. Avoid apologizing or downplaying your abilities, as this can undermine your          │
│  confidence and credibility.                                                                                    │
│                                                                                                                 │
│  • **Be honest and transparent about your experience and limitations**: It's okay to admit what you don't know  │
│  or haven't done before. Honesty and transparency are essential qualities for a Python Developer, and showing   │
│  willingness to learn and grow is attractive to interviewers.                                                   │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Checklist Writer                                                                              │
│                                                                                                                 │
│  Task: Using the researcher's tips, create a structured pre-interview checklist for Python Developer with       │
│  sections: Before the Interview, During the Interview, After the Interview.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Checklist Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the structured pre-interview checklist for Python Developer:                                           │
│  ```                                                                                                            │
│  **Before the Interview**                                                                                       │
│  =====================                                                                                          │
│                                                                                                                 │
│  * Review common interview questions and practice answering them confidently                                    │
│  * Prepare to explain complex concepts simply (e.g., technical ideas, projects)                                 │
│  * Research the company, role, and team to show enthusiasm and interest                                         │
│  * Review your portfolio and be prepared to discuss specific projects and accomplishments                       │
│  * Plan your outfit and make sure you look professional and put-together                                        │
│                                                                                                                 │
│  **During the Interview**                                                                                       │
│  =====================                                                                                          │
│                                                                                                                 │
│  * Maintain eye contact with the interviewer while answering questions                                          │
│  * Use positive body language (e.g., sit up straight, uncross arms)                                             │
│  * Avoid jargon and technical buzzwords; strive for clarity and simplicity in communication                     │
│  * Show enthusiasm and interest in the company and role by asking thoughtful questions                          │
│  * Be honest and transparent about your experience and limitations                                              │
│  * Don't apologize or downplay your skills; instead, highlight your strengths and accomplishments               │
│                                                                                                                 │
│  **After the Interview**                                                                                        │
│  =====================                                                                                          │
│                                                                                                                 │
│  * Send a thank-you note or email to the interviewer within 24 hours                                            │
│  * Reiterate your interest in the role and company                                                              │
│  * Follow up on any questions or concerns you may have had during the interview                                 │
│  * Review the interview process and identify areas for improvement (e.g., common questions, body language)      │
│  * Update your portfolio and prepare for future interviews by incorporating lessons learned from this           │
│  experience                                            


===== PRE-INTERVIEW CHECKLIST =====

Here is the structured pre-interview checklist for Python Developer:
```
**Before the Interview**

* Review common interview questions and practice answering them confidently
* Prepare to explain complex concepts simply (e.g., technical ideas, projects)
* Research the company, role, and team to show enthusiasm and interest
* Review your portfolio and be prepared to discuss specific projects and accomplishments
* Plan your outfit and make sure you look professional and put-together

**During the Interview**

* Maintain eye contact with the interviewer while answering questions
* Use positive body language (e.g., sit up straight, uncross arms)
* Avoid jargon and technical buzzwords; strive for clarity and simplicity in communication
* Show enthusiasm and interest in the company and role by asking thoughtful questions
* Be honest and transparent about your experience and limitations
* Don't apologize or downplay your skills; instead, highlight your st

##  Part 5: AI Interview Preparation Agent (live web search) — MAIN DELIVERABLE

This is the actual unique agent for this project. It uses **SerperDevTool** to fetch real-time, role-specific information — skills, questions, and topics — then generates a personalized **7-day preparation plan** for any given job role. Run on **2 different inputs** below to meet the minimum requirement.

Loads the Serper API key from the `.env` file (never hardcoded) and creates `search_tool`, which performs real Google searches to fetch current, role-specific interview information.

In [12]:
from dotenv import load_dotenv
from crewai_tools import SerperDevTool

load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

if not SERPER_API_KEY:
    raise ValueError(
        "SERPER_API_KEY not found. Create a .env file in this folder with:\n"
        "SERPER_API_KEY=your_free_key_from_serper.dev"
    )

os.environ["SERPER_API_KEY"] = SERPER_API_KEY
search_tool = SerperDevTool()

print("✅ Search tool ready")

✅ Search tool ready


Defines the two main agents: an `interview_researcher` that extracts verified, role-specific facts from live search results, and an `interview_coach` that creates a structured 7-day preparation plan from those facts.

In [13]:
interview_researcher = Agent(
    role="Interview Research Analyst",
    goal=(
        "Given web search results about {job_role} interviews, extract 6-10 concrete, "
        "factual details — required skills, commonly asked questions, important technical "
        "topics, and tools or technologies frequently mentioned."
    ),
    backstory=(
        "You are a meticulous career research analyst who reads job postings and "
        "interview guides carefully, reporting only verified, role-specific facts."
    ),
    tools=[],
    llm=ollama_llm,
    verbose=True,
    allow_delegation=False,
)

interview_coach = Agent(
    role="Interview Preparation Coach",
    goal=(
        "Using the researcher's findings, create a complete interview preparation "
        "plan for {job_role} covering a 7-day study plan, practice questions, and final tips."
    ),
    backstory=(
        "You are an experienced interview coach who has helped hundreds of candidates "
        "land their dream jobs by giving them structured, actionable preparation plans."
    ),
    tools=[],
    llm=ollama_llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Main agents ready")

✅ Main agents ready


Calls `search_tool` directly with a role-specific query and returns the results as plain text — this avoids tool-calling schema errors that local models sometimes encounter with automatic tool use.

In [14]:
def run_web_research(job_role):
    """Run a Serper web search for the job role and return results as plain text."""
    query = f"{job_role} interview questions skills topics 2024"
    raw_results = search_tool.run(search_query=query)
    return str(raw_results)

Defines both tasks (research + coaching) and packages them with the two agents into a `Crew`, injecting live search results into the research task so the coach works from real, current data.

In [15]:
def build_interview_crew(job_role, search_results_text):
    research_task = Task(
        description=(
            f"Job Role: {job_role}\n\n"
            f"Below are web search results:\n{search_results_text}\n\n"
            "From this, extract 6-10 concrete facts: required skills, commonly asked "
            "interview questions, key technical topics, and tools frequently mentioned."
        ),
        expected_output="A bullet list of 6-10 factual, role-specific interview preparation points.",
        agent=interview_researcher,
    )

    coach_task = Task(
        description=(
            f"Using the researcher's findings, create a complete interview preparation "
            f"plan for {job_role} with the following sections:\n"
            "- Role Overview\n"
            "- Key Skills & Topics to Study\n"
            "- Common Interview Questions (at least 5)\n"
            "- 7-Day Preparation Plan\n"
            "- Final Tips\n\n"
            "Output as clean, structured markdown."
        ),
        expected_output="A complete structured markdown interview preparation plan.",
        agent=interview_coach,
    )

    return Crew(
        agents=[interview_researcher, interview_coach],
        tasks=[research_task, coach_task],
        process=Process.sequential,
    )

### ✅ Minimum Requirement: Run on 2 Different Inputs
Loops through **Software Engineer** and **Data Analyst** — for each role, performs a live web search, builds the crew, and runs both agents to produce a complete interview preparation plan.

In [16]:
import asyncio

job_roles = [
    "Software Engineer",
    "Data Analyst"
]

async def run_interview_prep():
    for job_role in job_roles:
        print(f"\n{'='*60}\n💼 INTERVIEW PREP PLAN FOR: {job_role}\n{'='*60}\n")

        search_results_text = run_web_research(job_role)
        crew = build_interview_crew(job_role, search_results_text)
        result = await crew.kickoff_async(inputs={"job_role": job_role})

        final_text = getattr(result, "final_output", None) or getattr(result, "output", None) or result
        print(final_text)

await run_interview_prep()


💼 INTERVIEW PREP PLAN FOR: Software Engineer



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Research Analyst                                                                              │
│                                                                                                                 │
│  Task: Job Role: Software Engineer                                                                              │
│                                                                                                                 │
│  Below are web search results:                                                                                  │
│  {'searchParameters': {'q': 'Software Engineer interview questions skills topics 2024', 'type': 'search',       │
│  'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Entry-Level Software Engineer Interview Questions       │
│  (With Answers)', 'link':                                                                                       │
│  'https://masterscareers.brown.edu/blog/2024/02/07/entry-level-software-engineer-interview-questions-with-answ  │
│  ers/', 'snippet': 'Why do you want to go into software engineering? · Talk about a project you completed       │
│  successfully. · Tell me about a time you overcame a problem ...', 'position': 1}, {'title': 'Most Asked        │
│  Interview Questions for Software Engineers - YouTube', 'link': 'https://www.youtube.com/watch?v=rTvuBEDVhgU',  │
│  'snippet': 'These are the 15 most asked interview questions for software engineers. Master these questions to  │
│  ace your next tech interview.', 'position': 2}, {'title': 'What are some subtle screening questions to         │
│  separate serious ...', 'link':                                                                                 │
│  'https://www.reddit.com/r/SoftwareEngineering/comments/1cxgrgd/what_are_some_subtle_screening_questions_to/',  │
│  'snippet': 'Here\'s one example: "What do you think of functional programming?" The answer isn\'t important    │
│  per se, but if a candidate doesn\'t at least know ...', 'position': 3}, {'title': '30 common software          │
│  engineer interview questions for 2026 - Wiz', 'link':                                                          │
│  'https://www.wiz.io/academy/cloud-careers/software-engineer-interview-questions', 'snippet': 'The most         │
│  effective software engineer interviews combine coding assessments, system design discussions, and behavioral   │
│  scenarios tailored to the ...', 'position': 4}, {'title': 'Cracking The Mid-Level Software Engineer Interview  │
│  — Part I', 'link':                                                                                             │
│  'https://medium.com/@ortoren/cracking-the-mid-level-software-engineer-interview-part-i-4965916f619c',          │
│  'snippet': 'For example, I was asked questions about OOP, interfaces, dependency injection, SQL/NoSQL          │
│  databases, asynchronous programming, dockers, ...', 'position': 5}, {'title': '60 Engineer Interview           │
│  Questions & Answers - Intuit Blog', 'link':                                                                    │
│  'https://www.intuit.com/blog/innovative-thinking/60-engineering-interview-questions-to-expect-and-12-to-ask-d  │
│  uring-your-job-search/', 'snippet': 'On your previous projects, what problems or weaknesses were you able to   │
│  identify and solve? How did you work with a team to meet a shared goal?', 'position': 6}, {'title': 'Software  │
│  Engineer Interview Question Guide | Hiring on LinkedIn', 'link':                                               │
│  'https://business.linkedin.com/hire/resources/how-to-hire-guides/software-engineer-interview-questions',       │
│  'snippet': 'Top hard skills interview questions for hi

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Research Analyst                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the provided web search results, here are 7 concrete facts that can be used for software engineer     │
│  interview preparation:                                                                                         │
│                                                                                                                 │
│  • **Required skills:** The results suggest that proficiency in programming languages such as Python is         │
│  important. Additionally, knowledge of Agile methodology and experience with coding practices are also          │
│  mentioned.                                                                                                     │
│                                                                                                                 │
│  • **Commonly asked interview questions:**                                                                      │
│          + "Why do you want to go into software engineering?"                                                   │
│          + "Talk about a project you completed successfully."                                                   │
│          + "Tell me about a time you overcame a problem..."                                                     │
│          + "What is your quality assurance process for writing code and how do you spot a bug in an             │
│  application?"                                                                                                  │
│                                                                                                                 │
│  • **Key technical topics:**                                                                                    │
│          + OOP (Object-Oriented Programming)                                                                    │
│          + Interfaces                                                                                           │
│          + Dependency injection                                                                                 │
│          + SQL/NoSQL databases                                                                                  │
│          + Asynchronous programming                                                                             │
│          + Docker                                                                                               │
│                                                                                                                 │
│  • **Tools frequently mentioned:** None specifically mentioned, but the results suggest that proficiency in     │
│  coding practices and experience with Agile methodology are important.                                          │
│                                                                                                                 │
│  Note: These facts are based on the provided web search results and may not be exhaustive or representative of  │
│  all software engineer interviews.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Preparation Coach                                                                             │
│                                                                                                                 │
│  Task: Using the researcher's findings, create a complete interview preparation plan for Software Engineer      │
│  with the following sections:                                                                                   │
│  - Role Overview                                                                                                │
│  - Key Skills & Topics to Study                                                                                 │
│  - Common Interview Questions (at least 5)                                                                      │
│  - 7-Day Preparation Plan                                                                                       │
│  - Final Tips                                                                                                   │
│                                                                                                                 │
│  Output as clean, structured markdown.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Research Analyst                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the provided web search results, here are the concrete facts extracted:                               │
│                                                                                                                 │
│  • **Required skills:** Communication skills, problem-solving skills, teamwork, written and verbal              │
│  communication, excellent at writing queries, reports, and presentations.                                       │
│  • **Commonly asked interview questions:**                                                                      │
│          + Talk to us about “this” project — either from your portfolio or past experience                      │
│          + What could you have done to improve the project?                                                     │
│          + How do you present your findings to a non-technical audience?                                        │
│          + Can you describe a time when you had to explain a complex data analysis to a non-technical           │
│  audience?                                                                                                      │
│          + Why do you want to be a data analyst?                                                                │
│          + Tell me about a time you failed.                                                                     │
│  • **Key technical topics:**                                                                                    │
│          + SQL (Windows Functions, LAG, etc.)                                                                   │
│          + Data analysis                                                                                        │
│          + Data visualization                                                                                   │
│          + Communication of complex data analysis to non-technical audiences                                    │
│  • **Tools frequently mentioned:** None explicitly mentioned in the search results.                             │
│                                                                                                                 │
│  Here is the complete response:                                                                                 │
│                                                                                                                 │
│  • Required skills: Communication skills, problem-solving skills, teamwork, written and verbal communication,   │
│  excellent at writing queries, reports, and presentations.                                                      │
│  • Commonly asked interview questions:                                                                          │
│          + Talk to us about “this” project — either from your portfolio or past experience                      │
│          + What could you have done to improve the project?                                                     │
│          + How do you present your findings to a non-technical audience?                                        │
│          + Can you describe a time when you had to explain a complex data analysis to a non-technical           │
│  audience?                                                                                                      │
│          + Why do you want to be a data analyst?       

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Preparation Coach                                                                             │
│                                                                                                                 │
│  Task: Using the researcher's findings, create a complete interview preparation plan for Data Analyst with the  │
│  following sections:                                                                                            │
│  - Role Overview                                                                                                │
│  - Key Skills & Topics to Study                                                                                 │
│  - Common Interview Questions (at least 5)                                                                      │
│  - 7-Day Preparation Plan                                                                                       │
│  - Final Tips                                                                                                   │
│                                                                                                                 │
│  Output as clean, structured markdown.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Preparation Coach                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Data Analyst Interview Preparation Plan**                                                                    │
│  =====================================================                                                          │
│                                                                                                                 │
│  ### Role Overview                                                                                              │
│  A Data Analyst is responsible for analyzing and interpreting complex data to help organizations make informed  │
│  business decisions. They must be able to communicate their findings effectively to both technical and          │
│  non-technical audiences, and work collaboratively with cross-functional teams.                                 │
│                                                                                                                 │
│  ### Key Skills & Topics to Study                                                                               │
│  To succeed in a Data Analyst interview, you should focus on the following skills and topics:                   │
│                                                                                                                 │
│  * **Communication skills**: Ability to present complex data insights to non-technical audiences.               │
│  * **Problem-solving skills**: Capacity to analyze complex data sets and identify meaningful patterns.          │
│  * **Teamwork**: Willingness to collaborate with cross-functional teams.                                        │
│  * **Written and verbal communication**: Ability to write clear reports and present findings effectively.       │
│  * **Excellent at writing queries, reports, and presentations**: Proficiency in creating data visualizations    │
│  and presenting insights.                                                                                       │
│                                                                                                                 │
│  ### Common Interview Questions                                                                                 │
│  Here are five common interview questions for a Data Analyst position:                                          │
│                                                                                                                 │
│  1. **Talk to us about “this” project — either from your portfolio or past experience**:                        │
│          * Prepare to discuss a specific project you worked on, highlighting your role, responsibilities, and   │
│  accomplishments.                                                                                               │
│  2. **What could you have done to improve the project?**:                                                       │
│          * Be prepared to provide constructive feedback on how you could have improved the project's outcome    │
│  or process.                                                                                                    │
│  3. **How do you present your findings to a non-technical audience?**:                                          │
│          * Prepare examples of how you've effectively communicated complex data insights to non-technical       │
│  stakeholders.                                         

**Data Analyst Interview Preparation Plan**

### Role Overview
A Data Analyst is responsible for analyzing and interpreting complex data to help organizations make informed business decisions. They must be able to communicate their findings effectively to both technical and non-technical audiences, and work collaboratively with cross-functional teams.

### Key Skills & Topics to Study
To succeed in a Data Analyst interview, you should focus on the following skills and topics:

* **Communication skills**: Ability to present complex data insights to non-technical audiences.
* **Problem-solving skills**: Capacity to analyze complex data sets and identify meaningful patterns.
* **Teamwork**: Willingness to collaborate with cross-functional teams.
* **Written and verbal communication**: Ability to write clear reports and present findings effectively.
* **Excellent at writing queries, reports, and presentations**: Proficiency in creating data visualizations and presenting insights.

### Comm

## Conclusion

This notebook successfully demonstrates a **multi-agent AI Interview Preparation system** built using **CrewAI + Ollama (Llama 3) + SerperDevTool**.

### Key Points

- **Two Agents Work Together:** An `Interview Research Analyst` extracts verified, role-specific facts from live web search results, and an `Interview Preparation Coach` turns those facts into a complete, structured preparation plan.

- **Local LLM (No Cloud Cost):** The entire AI pipeline runs on **Llama 3 locally via Ollama** — no OpenAI API key or cloud charges needed for the model itself.

- **Live Web Search:** `SerperDevTool` fetches **real-time Google search results** for any given job role, ensuring the preparation plan reflects current industry requirements and interview trends.

- **Sequential Process:** `Process.sequential` ensures the researcher always completes first before the coach starts — because the coach depends entirely on the researcher's extracted facts.

- **2 Different Inputs Tested:** The final agent was successfully run on **"Software Engineer"** and **"Data Analyst"**, meeting the minimum project requirement.

- **Secure API Key Handling:** The Serper API key is loaded from a `.env` file instead of being hardcoded — a best practice for keeping credentials safe in submitted notebooks.

- **7-Day Preparation Plan:** Unlike generic interview guides, this agent generates a role-specific, day-by-day study plan tailored to the exact job role provided as input.

>  **Disclaimer:** All preparation plans are AI-generated summaries based on public web search results, produced for academic purposes only — not professional career advice.

